# New 仓库去重率测试

测试 New 仓库的两级去重机制在 Docker 镜像层数据上的去重效果。

| 级别 | 方法 | 原理 |
|------|------|------|
| **Tier 1** | File-as-Layer | tar 提取文件 → SHA256 精确匹配去重 |
| **Tier 2** | File+CNN Chunk | Tier 1 + 唯一大文件(≥512KiB) CNN 语义哈希 → DBSCAN 聚类 → fastcdc 子块去重 |

**Tier 2 的设计动机**：小文件通过文件级精确匹配已充分去重；大文件可能跨版本/跨镜像存在**近似相同**的区域，  
通过 CNN 语义哈希发现这些相似块，再用 DBSCAN 聚类 + fastcdc 捕获更细粒度的重复。

**运行环境**: AUTODL GPU 平台 (PyTorch + CUDA)

In [ ]:
import os
import sys
import tarfile
import gzip
import hashlib
import shutil
import gc
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ML 相关 (Tier 2 CNN 语义哈希需要)
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import DBSCAN

# 中文字体支持（Linux 环境）
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 库导入完成 | PyTorch device: {device}")

## 1. 配置：设置 Layer 数据目录

将 `layer_dir` 修改为存放 Docker layer blob 文件的目录路径。  
支持嵌套子目录，脚本会自动递归查找所有 gzip 文件。

In [ ]:
# ============================================================
# ⚙️ 配置区 — 根据 AUTODL 环境修改以下路径
# ============================================================

# --- Layer 数据目录 (存放 Docker layer .tar.gz 文件) ---
layer_dir = '/root/experiment/data/nginx_layers/'

# --- CNN 预训练模型路径 (Tier 2 语义哈希需要) ---
model_base = '/root/experiment/01/training/training/model/'
basemodel_path = os.path.join(model_base, 'model_data.torchsave')
hashmodel_path = os.path.join(model_base, 'model_hash.torchsave')

# --- 常量 ---
CHUNK_SIZE = 512 * 1024   # 512 KiB — CNN 模型输入尺寸
HASH_SIZE = 128           # 语义哈希位数

print(f"📁 Layer 目录:    {layer_dir}  (存在: {os.path.isdir(layer_dir)})")
print(f"🧠 Backbone 模型: {basemodel_path}  (存在: {os.path.isfile(basemodel_path)})")
print(f"🧠 Hash 模型:     {hashmodel_path}  (存在: {os.path.isfile(hashmodel_path)})")

## 2. 工具函数

定义查找 layer 文件和从 layer 中提取文件的函数。  
提取逻辑与 New 仓库 `j_dedup.go` 中的 `ExtractFilesFromLayer` 保持一致。

In [ ]:
def sha256_digest(data: bytes) -> str:
    """计算 SHA256 摘要"""
    return hashlib.sha256(data).hexdigest()


def find_layers(layer_dir: str) -> list:
    """
    递归查找目录下所有 gzip layer 文件。
    通过检查文件头部 magic bytes (0x1f 0x8b) 来判断是否为 gzip 文件。
    """
    layers = []
    for root, dirs, files in os.walk(layer_dir):
        for fname in sorted(files):
            fpath = os.path.join(root, fname)
            try:
                with open(fpath, 'rb') as f:
                    magic = f.read(2)
                if magic == b'\x1f\x8b':  # gzip magic number
                    layers.append(fpath)
            except:
                pass
    return layers


def extract_files_from_layer(layer_path: str) -> list:
    """
    从一个 tar.gz layer 中提取所有文件条目。
    与 01/new/registry/storage/j_dedup.go 中 ExtractFilesFromLayer 逻辑一致：
      - 普通文件: SHA256(content)
      - 符号链接: SHA256("symlink:" + target)
      - 硬链接:   SHA256("hardlink:" + target)
      - 目录:     标记 is_dir=True, 不计算内容 hash
    
    返回 list of dict: {path, digest, size, type, layer}
    """
    entries = []
    layer_name = os.path.basename(layer_path)
    try:
        with tarfile.open(layer_path, 'r:gz') as tar:
            for member in tar:
                if member.isdir():
                    entries.append({
                        'path': member.name,
                        'digest': None,
                        'size': 0,
                        'type': 'dir',
                        'layer': layer_name,
                    })
                elif member.isfile():
                    f = tar.extractfile(member)
                    if f is None:
                        continue
                    content = f.read()
                    digest = sha256_digest(content)
                    entries.append({
                        'path': member.name,
                        'digest': digest,
                        'size': len(content),
                        'type': 'file',
                        'layer': layer_name,
                    })
                elif member.issym():
                    pseudo = f"symlink:{member.linkname}".encode()
                    digest = sha256_digest(pseudo)
                    entries.append({
                        'path': member.name,
                        'digest': digest,
                        'size': 0,
                        'type': 'symlink',
                        'layer': layer_name,
                    })
                elif member.islnk():
                    pseudo = f"hardlink:{member.linkname}".encode()
                    digest = sha256_digest(pseudo)
                    entries.append({
                        'path': member.name,
                        'digest': digest,
                        'size': 0,
                        'type': 'hardlink',
                        'layer': layer_name,
                    })
                else:
                    # whiteout 等特殊文件
                    pseudo = f"special:{member.name}".encode()
                    digest = sha256_digest(pseudo)
                    entries.append({
                        'path': member.name,
                        'digest': digest,
                        'size': 0,
                        'type': 'special',
                        'layer': layer_name,
                    })
    except Exception as e:
        print(f"  ⚠️  无法解析 {layer_path}: {e}")
    return entries


print("✅ 工具函数定义完成")

## 3. 扫描并提取所有 Layer 的文件

查找所有 layer 文件，逐个打开并提取其中的文件条目。

In [ ]:
# 查找所有 layer 文件
layers = find_layers(layer_dir)
print(f"📦 发现 {len(layers)} 个 layer 文件\n")

if len(layers) == 0:
    raise FileNotFoundError(f"未在 {layer_dir} 下找到任何 gzip layer 文件")

# 提取所有文件条目
all_entries = []
layer_meta = []  # 记录每层的元信息

for i, lpath in enumerate(layers):
    entries = extract_files_from_layer(lpath)
    all_entries.extend(entries)
    
    lname = os.path.basename(lpath)
    layer_size = os.path.getsize(lpath)
    num_files = sum(1 for e in entries if e['type'] == 'file')
    
    layer_meta.append({
        'layer': lname,
        'compressed_size': layer_size,
        'total_entries': len(entries),
        'file_entries': num_files,
    })
    
    print(f"  [{i+1:3d}/{len(layers)}] {lname[:20]}...  "
          f"{num_files} 个文件, "
          f"压缩 {layer_size/1024/1024:.2f} MiB")

# 过滤出有 digest 的条目（排除目录）
all_files = [e for e in all_entries if e['digest'] is not None]

print(f"\n✅ 共提取 {len(all_entries)} 个条目, 其中 {len(all_files)} 个有效文件")

## 4. 文件级去重率统计

按 SHA256 digest 对所有文件去重，计算：
- 总文件数 / 唯一文件数
- 总大小 / 唯一大小 / 节省大小
- **文件级去重率** = 总大小 / 唯一大小

In [ ]:
# ---------- 文件级去重统计 ----------
from collections import defaultdict

file_map = defaultdict(lambda: {"count": 0, "size": 0, "paths": []})

for entry in all_files:
    d = entry["digest"]
    file_map[d]["count"] += 1
    file_map[d]["size"] = entry["size"]          # 同 digest 大小相同
    file_map[d]["paths"].append(entry["path"])

total_file_count = sum(v["count"] for v in file_map.values())
unique_file_count = len(file_map)

total_file_size = sum(v["count"] * v["size"] for v in file_map.values())
unique_file_size = sum(v["size"] for v in file_map.values())
saved_size = total_file_size - unique_file_size

dedup_ratio = total_file_size / unique_file_size if unique_file_size > 0 else 1.0

print(f"=== 文件级去重 (File-as-Layer) ===")
print(f"总文件数:       {total_file_count:>12,}")
print(f"唯一文件数:     {unique_file_count:>12,}")
print(f"重复文件数:     {total_file_count - unique_file_count:>12,}")
print()
print(f"总文件大小:     {total_file_size:>12,} bytes  ({total_file_size/1024**3:.2f} GiB)")
print(f"唯一文件大小:   {unique_file_size:>12,} bytes  ({unique_file_size/1024**3:.2f} GiB)")
print(f"节省大小:       {saved_size:>12,} bytes  ({saved_size/1024**3:.2f} GiB)")
print()
print(f"★ 文件级去重率: {dedup_ratio:.4f}x")
print(f"★ 空间节省率:   {saved_size/total_file_size*100:.2f}%" if total_file_size > 0 else "")

## 5. Top-10 重复最多的文件

按节省字节数 `size × (count - 1)` 降序排列，展示重复收益最大的文件。

In [ ]:
# ---------- Top-10 重复文件 ----------
import pandas as pd

top_files = sorted(
    file_map.items(),
    key=lambda kv: kv[1]["size"] * (kv[1]["count"] - 1),
    reverse=True,
)[:10]

rows = []
for digest, info in top_files:
    saved = info["size"] * (info["count"] - 1)
    sample_path = info["paths"][0]  # 取第一个路径作为示例
    rows.append({
        "digest (前16字符)": digest[:16] + "…",
        "文件大小": f"{info['size']:,}",
        "出现次数": info["count"],
        "节省字节": f"{saved:,}",
        "示例路径": sample_path,
    })

df_top = pd.DataFrame(rows)
df_top.index = range(1, len(df_top) + 1)
df_top.index.name = "排名"
display(df_top)

## 6. 每 Layer 去重贡献分析

统计每个 Layer 的文件数、唯一文件数、与其他 Layer 共享的文件数。

In [ ]:
# ---------- 每 Layer 去重贡献 ----------

# 按 layer 分组
layer_files = defaultdict(list)
for entry in all_files:
    layer_files[entry["layer"]].append(entry)

rows = []
seen_global = set()  # 全局已见 digest（按处理顺序累计）

for layer_name in sorted(layer_files.keys()):
    entries = layer_files[layer_name]
    digests = [e["digest"] for e in entries]
    unique_in_layer = set(digests)

    total_size = sum(e["size"] for e in entries)
    unique_size = sum(
        next(e["size"] for e in entries if e["digest"] == d)
        for d in unique_in_layer
    )
    new_digests = unique_in_layer - seen_global
    shared_digests = unique_in_layer & seen_global

    rows.append({
        "Layer": layer_name[:16] + ("…" if len(layer_name) > 16 else ""),
        "文件数": len(entries),
        "层内唯一": len(unique_in_layer),
        "跨层新增": len(new_digests),
        "跨层共享": len(shared_digests),
        "总大小 (KiB)": round(total_size / 1024, 1),
        "唯一大小 (KiB)": round(unique_size / 1024, 1),
    })
    seen_global |= unique_in_layer

df_layers = pd.DataFrame(rows)
display(df_layers)

print(f"\n共扫描 {len(layer_files)} 个 Layer")

## 7. 可视化

- 左图：总大小 vs 唯一大小 柱状图
- 右图：去重 / 唯一 占比饼图

In [ ]:
# ---------- 可视化 ----------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- 柱状图 ---
ax = axes[0]
labels = ["Total Size", "Unique Size"]
sizes_gib = [total_file_size / 1024**3, unique_file_size / 1024**3]
colors = ["#4C72B0", "#55A868"]
bars = ax.bar(labels, sizes_gib, color=colors, width=0.5)
for bar, val in zip(bars, sizes_gib):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.2f} GiB", ha="center", va="bottom", fontsize=11)
ax.set_ylabel("Size (GiB)")
ax.set_title(f"File-level Dedup  (ratio = {dedup_ratio:.2f}x)")
ax.set_ylim(0, max(sizes_gib) * 1.15)

# --- 饼图 ---
ax2 = axes[1]
pie_labels = ["Duplicated", "Unique"]
pie_sizes = [saved_size, unique_file_size]
pie_colors = ["#C44E52", "#55A868"]
wedges, texts, autotexts = ax2.pie(
    pie_sizes, labels=pie_labels, colors=pie_colors,
    autopct="%1.1f%%", startangle=90, textprops={"fontsize": 11}
)
ax2.set_title("Storage Composition")

plt.tight_layout()
plt.savefig("file_dedup_ratio.png", dpi=150, bbox_inches="tight")
plt.show()
print("图表已保存为 file_dedup_ratio.png")

## 8. CNN 语义哈希模型 + 辅助函数

Tier 2 (File+CNN Chunk) 需要 CNN 语义哈希来发现大文件中的近似重复块：

1. **RevisedNetwork**：10层 Conv1D backbone（与 SimEnc 论文架构一致）
2. **HashNetwork**：backbone → fc_plus → **128-bit 语义哈希**
3. **GreedyHashLoss**：训练时使用的损失函数（推理时仅需 `Hash.apply`）
4. **batch_semantic_hash()**：批量 GPU 推理接口
5. **get_cdc_chunks()**：Content-Defined Chunking 变长分块

In [ ]:
# ===== CNN 模型定义 (与 SimEnc 论文架构一致) =====
_which_dense = 2
_hashSize = HASH_SIZE       # 128
_denseSize1 = 4096
_denseSize2 = 2048
numCluster = 261            # 必须与预训练模型匹配

class GreedyHashLoss(torch.nn.Module):
    def __init__(self, bit, alpha=1):
        super().__init__()
        self.fc = nn.Linear(bit, numCluster, bias=False).to(device)
        self.criterion = nn.CrossEntropyLoss().to(device)
        self.alpha = alpha

    def forward(self, outputs, y, feature):
        loss1 = self.criterion(outputs, y)
        loss2 = self.alpha * (feature.abs() - 1).pow(3).abs().mean()
        return loss1 + loss2

    class Hash(torch.autograd.Function):
        @staticmethod
        def forward(ctx, input):
            return input.sign()
        @staticmethod
        def backward(ctx, grad_output):
            return grad_output

class RevisedNetwork(nn.Module):
    """Backbone CNN: 10层 Conv1D + 2层 Dense"""
    def __init__(self):
        super().__init__()
        conv = []
        channels = [1, 8, 16, 32, 32, 32, 32, 32, 32, 32, 32]
        for i in range(10):
            conv += [
                nn.Conv1d(channels[i], channels[i+1], 3, 1, 1, bias=True),
                nn.ReLU(),
                nn.BatchNorm1d(channels[i+1]),
                nn.MaxPool1d(2),
            ]
        self.conv_layers = nn.ModuleList(conv)
        layers = [nn.Linear(512 * 32, _denseSize1), nn.ReLU(), nn.Dropout(0.5)]
        if _denseSize2 > 0:
            layers += [nn.Linear(_denseSize1, _denseSize2), nn.ReLU(), nn.Dropout(0.5)]
        self.layers = nn.ModuleList(layers)
        self.fc = nn.Linear(_denseSize2 if _denseSize2 > 0 else _denseSize1,
                            numCluster, bias=False)

    def forward(self, x):
        x = x.unsqueeze(dim=1)
        for l in self.conv_layers:
            x = l(x)
        x = x.view(x.shape[0], -1)
        for l in self.layers:
            x = l(x)
        return F.log_softmax(self.fc(x), dim=1)

class HashNetwork(nn.Module):
    """Hash 网络: backbone 输出 → 128-bit 语义哈希"""
    def __init__(self, revNet):
        super().__init__()
        self.conv_layers = revNet.conv_layers
        self.layers = revNet.layers
        last = _denseSize2 if (_denseSize2 > 0 and _which_dense == 2) else _denseSize1
        self.fc_plus = nn.Linear(last, _hashSize)
        self.fc = nn.Linear(_hashSize, numCluster, bias=False)

    def forward(self, x):
        x = x.unsqueeze(dim=1)
        for l in self.conv_layers:
            x = l(x)
        x = x.view(x.shape[0], -1)
        if _which_dense == 2:
            for l in self.layers:
                x = l(x)
        else:
            x = self.layers[0](x)
        x = self.fc_plus(x)
        code = GreedyHashLoss.Hash.apply(x)
        output = self.fc(code)
        return output, x, code

# --- 加载预训练模型 ---
print("加载预训练 CNN 模型 ...")
basemodel = RevisedNetwork()
basemodel.load_state_dict(torch.load(basemodel_path, map_location=device))
hidden_model = HashNetwork(basemodel)
hidden_model.load_state_dict(torch.load(hashmodel_path, map_location=device))
hidden_model = hidden_model.to(device)
hidden_model.eval()
print(f"✅ CNN 模型加载完成 → {device}")

# --- 辅助函数: CDC 变长分块 ---
from fastcdc import fastcdc as fastcdc_chunker

def get_cdc_chunks(filepath, min_size=256, avg_size=512, max_size=1024):
    """对文件执行 Content-Defined Chunking，返回 [(hash, size), ...]"""
    results = []
    for chunk in fastcdc_chunker(filepath, min_size, avg_size, max_size,
                                  fat=True, hf=hashlib.sha256):
        results.append((chunk.hash, chunk.length))
    return results

# --- 辅助函数: 批量 CNN 推理 ---
def batch_semantic_hash(filenames, batch_size=400 if torch.cuda.is_available() else 50):
    """对一批 512KiB chunk 文件计算 128-bit 语义哈希"""
    all_hashes = []
    for start in range(0, len(filenames), batch_size):
        batch_files = filenames[start:start+batch_size]
        batch_data = []
        for fn in batch_files:
            with open(fn, 'rb') as f:
                data = f.read()
            t = torch.tensor([int(b) for b in data], dtype=torch.float32)
            t = (t - 128) / 128.0  # 归一化到 [-1, 1]
            batch_data.append(t.unsqueeze(0))
        
        batch_tensor = torch.cat(batch_data, dim=0).to(device)
        with torch.no_grad():
            _, feature, code = hidden_model(batch_tensor)
        
        for c in code:
            all_hashes.append(c.cpu())
        
        del batch_tensor, feature, code
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        
        print(f"  推理进度: {min(start+batch_size, len(filenames))}/{len(filenames)}", end='\r')
    
    return all_hashes

print("✅ 辅助函数定义完成")

## 9. Tier 2 — File+CNN Chunk 去重

在文件级 SHA256 去重 (Tier 1) 的基础上，对 **≥512KiB 的唯一大文件** 进一步执行：
1. 按 512KiB 固定块切分 + zero-padding
2. **CNN 语义哈希**（128-bit hash）
3. **DBSCAN 聚类**（Hamming distance, eps=0.03 ≈ 4 bit 差异）
4. **Cluster 内 fastcdc 变长分块去重**

> 小文件（< 512KiB）已被文件级精确匹配充分覆盖，不再切块。  
> 大文件的 CNN 语义哈希可发现跨版本/跨镜像的 **近似相同** 512KiB 区域。  
> 对应 Go 代码 `FileDedup()` 中的 **Step 2.5**。

In [ ]:
# ===== Tier 2: File+CNN Chunk =====
# 复用 Tier 1 的 file_map 文件级去重结果，
# 对唯一大文件 (>= 512KiB) 做 CNN 语义哈希 + DBSCAN + cluster 内 fastcdc 去重。

# ========== Step 1: 收集唯一大文件 ==========
print("Step 1/5: 收集唯一大文件 (>= 512KiB) ...")
unique_large_digests = {}
for dgst, info in file_map.items():
    if info["size"] >= CHUNK_SIZE:
        unique_large_digests[dgst] = info

print(f"  唯一大文件: {len(unique_large_digests)} 个"
      f" (占唯一文件 {len(unique_large_digests)/max(len(file_map),1)*100:.1f}%)")

# ========== Step 2: 从 layer 提取大文件内容 ==========
print("Step 2/5: 从 layer 提取大文件内容 ...")
large_file_contents = {}
for i, lp in enumerate(layers):
    layer_name = os.path.basename(lp)
    try:
        with tarfile.open(lp, 'r:gz') as tar:
            for member in tar:
                if not member.isfile():
                    continue
                f = tar.extractfile(member)
                if f is None:
                    continue
                content = f.read()
                dgst = sha256_digest(content)
                if dgst in unique_large_digests and dgst not in large_file_contents:
                    large_file_contents[dgst] = content
    except Exception as e:
        print(f"  ⚠️ 无法读取 {layer_name}: {e}")
    print(f"  [{i+1}/{len(layers)}] 已扫描 {layer_name[:20]}...", end='\r')

print(f"\n  提取到 {len(large_file_contents)} 个大文件的内容")

# ========== Step 3: 切成 512KiB chunk → zero-padding → CNN 推理 ==========
print("Step 3/5: 切块 + zero-padding → CNN 语义哈希推理 ...")
new_chunk_dir = os.path.join(os.path.dirname(layer_dir), "new_cnn_chunks")
if os.path.isdir(new_chunk_dir):
    shutil.rmtree(new_chunk_dir)
os.makedirs(new_chunk_dir, exist_ok=True)

new_chunk_files = []
for dgst, content in large_file_contents.items():
    for offset in range(0, len(content), CHUNK_SIZE):
        chunk = content[offset:offset + CHUNK_SIZE]
        if len(chunk) < CHUNK_SIZE:
            chunk += b'\x00' * (CHUNK_SIZE - len(chunk))  # zero-padding to 512KiB
        cp = os.path.join(new_chunk_dir, f'{dgst[:16]}_b{offset // CHUNK_SIZE}.bin')
        with open(cp, 'wb') as out:
            out.write(chunk)
        new_chunk_files.append(cp)

large_file_contents = None  # free memory
gc.collect()
print(f"  生成 {len(new_chunk_files)} 个 512KiB chunk → {new_chunk_dir}")

# CNN 推理
if len(new_chunk_files) > 0:
    new_hashes = batch_semantic_hash(new_chunk_files)
else:
    new_hashes = []

new_hash_df = pd.DataFrame({
    'file': new_chunk_files,
    'hash': new_hashes,
})
print(f"\n✅ CNN 语义哈希推理完成: {len(new_hash_df)} 个 chunk → 128-bit hash")

# ========== Step 4: DBSCAN 聚类 (Hamming distance) ==========
print("Step 4/5: DBSCAN 聚类 (Hamming distance, eps=0.03) ...")
if len(new_hash_df) > 1:
    new_hash_np = [((h + 1) / 2).detach().numpy() for h in new_hash_df['hash']]
    new_hamming_dist = pdist(new_hash_np, metric='hamming')
    new_hamming_matrix = squareform(new_hamming_dist).astype(np.float16)

    new_dbscan = DBSCAN(metric='precomputed', eps=0.03, min_samples=1, n_jobs=-1)
    new_dbscan.fit(new_hamming_matrix)
    new_hash_df['cluster'] = new_dbscan.labels_

    del new_hamming_dist, new_hamming_matrix
    gc.collect()
elif len(new_hash_df) == 1:
    new_hash_df['cluster'] = [0]
else:
    new_hash_df['cluster'] = []

new_cluster_counts = new_hash_df['cluster'].value_counts()
print(f"  ✅ 聚类完成: {len(new_cluster_counts)} 个 cluster")
if len(new_cluster_counts) > 0:
    print(f"     最大 cluster: {new_cluster_counts.iloc[0]} 个 chunk")
    print(f"     单元素 cluster: {(new_cluster_counts == 1).sum()} 个")

# ========== Step 5: Cluster 内 fastcdc 子块去重 ==========
print("\nStep 5/5: Cluster 内 fastcdc 子块去重 ...")
cnn_chunk_total_size = 0
cnn_chunk_unique_size = 0
num_new_clusters = len(new_cluster_counts)

for ci, cluster_id in enumerate(new_cluster_counts.index):
    cluster_files = new_hash_df[new_hash_df['cluster'] == cluster_id]['file'].tolist()
    cluster_hashes = {}  # hash → size (cluster 内去重)
    for f in cluster_files:
        cdc = get_cdc_chunks(f)
        for h, s in cdc:
            cnn_chunk_total_size += s
            if h not in cluster_hashes:
                cluster_hashes[h] = s
    cnn_chunk_unique_size += sum(cluster_hashes.values())
    if (ci + 1) % 20 == 0 or ci + 1 == num_new_clusters:
        print(f"  Cluster 进度: {ci+1}/{num_new_clusters}", end='\r')

# ========== 组合计算 ==========
small_file_total = sum(v["count"] * v["size"] for v in file_map.values() if v["size"] < CHUNK_SIZE)
small_file_unique = sum(v["size"] for v in file_map.values() if v["size"] < CHUNK_SIZE)
large_file_total_by_count = sum(v["count"] * v["size"] for v in file_map.values() if v["size"] >= CHUNK_SIZE)

combined_total = small_file_total + large_file_total_by_count
combined_unique = small_file_unique + cnn_chunk_unique_size
combined_saved = combined_total - combined_unique
combined_ratio = combined_total / combined_unique if combined_unique > 0 else 1.0

# CNN 对大文件的额外收益 (对比 Tier 1)
large_file_unique_fileonly = sum(v["size"] for v in file_map.values() if v["size"] >= CHUNK_SIZE)
cnn_extra_saved = large_file_unique_fileonly - cnn_chunk_unique_size

print(f"\n\n=== Tier 2: File+CNN Chunk ===")
print(f"--- 小文件 (< 512KiB) 文件级去重 ---")
print(f"  总大小:       {small_file_total:>12,} bytes")
print(f"  唯一大小:     {small_file_unique:>12,} bytes")
print(f"--- 大文件 (>= 512KiB) CNN + DBSCAN + fastcdc ---")
print(f"  唯一大文件数: {len(unique_large_digests):>12,}")
print(f"  总chunk数:    {len(new_chunk_files):>12,}")
print(f"  DBSCAN cluster: {len(new_cluster_counts):>10,}")
print(f"  chunk总大小:  {cnn_chunk_total_size:>12,} bytes")
print(f"  chunk唯一:    {cnn_chunk_unique_size:>12,} bytes")
print(f"  CNN额外节省:  {cnn_extra_saved:>12,} bytes")
print(f"--- 组合结果 ---")
print(f"  总大小:       {combined_total:>12,} bytes  ({combined_total/1024**3:.2f} GiB)")
print(f"  唯一大小:     {combined_unique:>12,} bytes  ({combined_unique/1024**3:.2f} GiB)")
print(f"  节省大小:     {combined_saved:>12,} bytes  ({combined_saved/1024**3:.2f} GiB)")
print(f"★ Tier 2 去重率: {combined_ratio:.4f}x")
print(f"★ 空间节省率:    {combined_saved/combined_total*100:.2f}%" if combined_total > 0 else "")
print(f"  (vs Tier 1 {dedup_ratio:.4f}x, CNN额外提升 {combined_ratio - dedup_ratio:.4f}x)")

## 10. Tier 1 vs Tier 2 去重率对比

对比 **Tier 1 (File-as-Layer)** 纯文件级去重与 **Tier 2 (File+CNN Chunk)** 加入 CNN 语义哈希后的去重效果。

> Tier 2 的额外收益来自 CNN 语义哈希对大文件近似重复块的发现能力。

In [ ]:
# ===== Tier 1 vs Tier 2 对比汇总 =====

comparison = pd.DataFrame({
    "方法": [
        "Tier 1: File-as-Layer",
        "Tier 2: File+CNN Chunk",
    ],
    "去重粒度": [
        "文件级 SHA256",
        "文件级 + 大文件 CNN 语义块",
    ],
    "总大小 (GiB)": [
        round(total_file_size / 1024**3, 4),
        round(combined_total / 1024**3, 4),
    ],
    "唯一大小 (GiB)": [
        round(unique_file_size / 1024**3, 4),
        round(combined_unique / 1024**3, 4),
    ],
    "去重率 (x)": [
        round(dedup_ratio, 4),
        round(combined_ratio, 4),
    ],
    "空间节省率 (%)": [
        round(saved_size / total_file_size * 100, 2) if total_file_size > 0 else 0,
        round(combined_saved / combined_total * 100, 2) if combined_total > 0 else 0,
    ],
})
display(comparison)

# --- Tier 2 详细指标 ---
small_count = sum(1 for v in file_map.values() if v["size"] < CHUNK_SIZE)
detail = pd.DataFrame({
    "指标": [
        "唯一文件总数",
        "  其中小文件 (< 512KiB)",
        "  其中大文件 (>= 512KiB)",
        "大文件占比 (按字节)",
        "大文件 CNN chunk 数",
        "DBSCAN cluster 数",
        "CNN 额外节省",
        "去重率提升 (Tier 2 − Tier 1)",
    ],
    "值": [
        f"{len(file_map):,}",
        f"{small_count:,}",
        f"{len(unique_large_digests):,}",
        f"{large_file_unique_fileonly / unique_file_size * 100:.1f}%" if unique_file_size > 0 else "0%",
        f"{len(new_chunk_files):,}",
        f"{len(new_cluster_counts):,}",
        f"{cnn_extra_saved:,} bytes ({cnn_extra_saved/1024**2:.2f} MiB)",
        f"+{combined_ratio - dedup_ratio:.4f}x",
    ],
})
display(detail)

# --- 对比可视化 ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 左图: Total vs Unique Size
ax = axes[0]
x = np.arange(2)
width = 0.3
bars1 = ax.bar(x - width/2, comparison["总大小 (GiB)"], width, label="Total", color="#4C72B0")
bars2 = ax.bar(x + width/2, comparison["唯一大小 (GiB)"], width, label="Unique", color="#55A868")
ax.set_xticks(x)
ax.set_xticklabels(["Tier 1\nFile-as-Layer", "Tier 2\nFile+CNN Chunk"], fontsize=9)
ax.set_ylabel("Size (GiB)")
ax.set_title("Total vs Unique Size")
ax.legend()
for b in list(bars1) + list(bars2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(),
            f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=9)

# 中图: 去重率对比
ax2 = axes[1]
colors = ["#55A868", "#8172B2"]
bars = ax2.bar(["Tier 1\nFile-as-Layer", "Tier 2\nFile+CNN Chunk"],
               comparison["去重率 (x)"], color=colors, width=0.4)
for b in bars:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height(),
             f"{b.get_height():.2f}x", ha="center", va="bottom", fontsize=12, fontweight='bold')
ax2.set_ylabel("Dedup Ratio (x)")
ax2.set_title("New 仓库去重率对比")

# 右图: 唯一文件大小分布 (小文件 vs 大文件)
ax3 = axes[2]
small_bytes = sum(v["size"] for v in file_map.values() if v["size"] < CHUNK_SIZE)
large_bytes = large_file_unique_fileonly
if (small_bytes + large_bytes) > 0:
    labels_pie = [f"小文件 (<512K)\n{small_count}个\n{small_bytes/1024**2:.1f} MiB",
                  f"大文件 (≥512K)\n{len(unique_large_digests)}个\n{large_bytes/1024**2:.1f} MiB"]
    wedges, texts, autotexts = ax3.pie(
        [small_bytes, large_bytes], labels=labels_pie,
        colors=["#55A868", "#8172B2"],
        autopct="%1.1f%%", startangle=90, textprops={"fontsize": 9}
    )
ax3.set_title("唯一文件大小分布\n(Tier 2 大文件部分受益于 CNN)")

plt.tight_layout()
plt.savefig("new_dedup_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 图表已保存为 new_dedup_comparison.png")

## 11. 导出结果到 CSV

将 New 仓库去重分析结果导出为 CSV 文件，方便撰写论文。

In [ ]:
# ---------- 导出 CSV ----------

# 1. Tier 1 vs Tier 2 对比表
comparison.to_csv("new_dedup_comparison.csv", index=False)
print("✅ new_dedup_comparison.csv 已保存")

# 2. 每 Layer 文件级分析表
df_layers.to_csv("per_layer_breakdown.csv", index=False)
print("✅ per_layer_breakdown.csv 已保存")

# 3. 文件去重映射表
all_digests = []
for digest, info in file_map.items():
    all_digests.append({
        "digest": digest,
        "size": info["size"],
        "count": info["count"],
        "saved_bytes": info["size"] * (info["count"] - 1),
        "is_large": info["size"] >= CHUNK_SIZE,
        "sample_path": info["paths"][0],
    })
df_all = pd.DataFrame(all_digests)
df_all.sort_values("saved_bytes", ascending=False, inplace=True)
df_all.to_csv("file_digest_map.csv", index=False)
print(f"✅ file_digest_map.csv 已保存 ({len(df_all)} 条)")

# 4. Tier 2 CNN chunk + cluster 详情
tier2_detail = pd.DataFrame({
    "指标": [
        "唯一大文件数 (>=512KiB)",
        "大文件 chunk 总数",
        "DBSCAN cluster 数",
        "CNN chunk 总字节 (fastcdc)",
        "CNN chunk 唯一字节 (fastcdc)",
        "CNN 额外节省字节 (vs Tier 1)",
        "小文件总字节 (文件级)",
        "小文件唯一字节 (文件级)",
        "组合总字节",
        "组合唯一字节",
        "Tier 2 去重率",
    ],
    "值": [
        len(unique_large_digests),
        len(new_chunk_files),
        len(new_cluster_counts),
        cnn_chunk_total_size,
        cnn_chunk_unique_size,
        cnn_extra_saved,
        small_file_total,
        small_file_unique,
        combined_total,
        combined_unique,
        f"{combined_ratio:.4f}x",
    ],
})
tier2_detail.to_csv("tier2_cnn_chunk_detail.csv", index=False)
print("✅ tier2_cnn_chunk_detail.csv 已保存")

# 5. CNN chunk cluster 映射
if len(new_hash_df) > 0:
    new_hash_df[['file', 'cluster']].to_csv("tier2_cnn_clusters.csv", index=False)
    print(f"✅ tier2_cnn_clusters.csv 已保存 ({len(new_cluster_counts)} clusters)")
else:
    print("⚠️ 无大文件 chunk，跳过 tier2_cnn_clusters.csv")

print("\n🎉 全部完成! New 仓库去重率测试数据已导出。")